# **Libraries & Functions** 

In [1]:
# !pip install --q --trusted-host pypi.org --trusted-host pypi.python.org --trusted-host files.pythonhosted.org seaborn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import itertools
import warnings
warnings.filterwarnings("ignore")
from dateutil.relativedelta import relativedelta
# Supervivencia SemiParametrica
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

# Group K-Fold Cross-Validation por Cuentas
from sklearn.model_selection import GroupKFold

from scipy.interpolate import interp1d

In [3]:
def asignar_tasa_mercado(row):
    plazo=row["PlazoContractual"]
    if plazo <= 30:
        return row["TasaMercado30d"]
    elif 31<=plazo<=180:
        return row["TasaMercado31_180d"]
    elif 181<=plazo<=360:
        return row["TasaMercado181_360d"]
    else:
        return row["TasaMercadoMas361d"]

In [4]:
def get_tt_asignada(date,ccy,x_new):
    if ccy=="PEN":
        data = data_tt_mn.copy()
    else:
        data = data_tt_me.copy()
    data_date_tt = data.loc[data.index==date,:]
    x = data_date_tt.columns[:].array
    y = data_date_tt.iloc[0,:].values
    f_interp = interp1d(x, y, kind='linear',fill_value='extrapolate')
    tt = f_interp(x_new) 

    return tt

# **I. Data Reading** 

In [5]:
path_rdm=r"D:\Validaciones\2025\Mesa\ALM&Liq\Q2\MCR_MIBPE_DPF_2025\1. Documentos recibidos\RdM\2da Entrega\1.Datos"
file_rdm="mb_dpf_mod_agg.csv"
data_rdm=pd.read_csv(fr"{path_rdm}\{file_rdm}")
for columna in data_rdm.columns:
    if "FECHA" in columna.upper():
        data_rdm[columna]=pd.to_datetime(data_rdm[columna])

# Get characteristics
print(data_rdm.shape) # (1412755, 15)
print(f"Cantidad de Operaciones: {data_rdm.Cliente.count()}") # 1412696
print(f"Cantidad de Clientes Únicos: {data_rdm.Cliente.unique().size}") # 246387
print(f"Cantidad de Cuentas Únicas: {data_rdm.Cuenta.unique().size}") # 789246
print(f"Suma de Capital Activo: {data_rdm[data_rdm.FlagStatus==0]['Capital'].sum()}") # 8846280447.73

data_bmk = data_rdm.copy()

(1412696, 15)
Cantidad de Operaciones: 1412696
Cantidad de Clientes Únicos: 246387
Cantidad de Cuentas Únicas: 789246
Suma de Capital Activo: 8846280447.73


In [6]:
# Data Curva TT
path_tt = fr"D:\Validaciones\2025\Mesa\ALM&Liq\Q2\MCR_MIBPE_DPF_2025\3. Resultados\Implementacion\Asignado"
data_tt_mn = pd.read_excel(fr"{path_tt}\Plantilla TTs Historico MB - Jun 25.xlsx",sheet_name="TTPasiva_MN",parse_dates=[0])
data_tt_mn.set_index("FECHA",inplace=True)
data_tt_mn = data_tt_mn.reindex(pd.date_range(start=data_tt_mn.index.min(),end=data_tt_mn.index.max(),freq="D")).fillna(method="ffill")

data_tt_me = pd.read_excel(fr"{path_tt}\Plantilla TTs Historico MB - Jun 25.xlsx",sheet_name="TTPasiva_ME",parse_dates=[0])
data_tt_me.set_index("FECHA",inplace=True)
data_tt_me = data_tt_mn.reindex(pd.date_range(start=data_tt_mn.index.min(),end=data_tt_mn.index.max(),freq="D")).fillna(method="ffill")

In [7]:
# Data Modelo Conductual - Propuesta
path_tt = fr"D:\Validaciones\2025\Mesa\ALM&Liq\Q2\MCR_MIBPE_DPF_2025\3. Resultados\Implementacion\Asignado"
data_model_prop = pd.read_csv(fr"{path_tt}\Supuesto_Conductual_DPF_Propuesta.csv")

# Data Modelo Conductual - Vigente
path_tt = fr"D:\Validaciones\2025\Mesa\ALM&Liq\Q2\MCR_MIBPE_DPF_2025\3. Resultados\Implementacion\Asignado"
data_model_vig = pd.read_csv(fr"{path_tt}\Supuesto_Conductual_DPF_Vigente.csv")

# **II. Engine Execution**

In [22]:
# Parametros
mes = pd.Timestamp(day=31,month=3,year=2025)
# Cuentas Activos: El Flag es 0 porque son los depósitos activos; Flag RETAIL porque Tesorería no puede cobrarse a sí mismo (IFIS, No IFIS)
# saldos_activos = data_bmk[(data_bmk["FlagStatus"]==0) & (data_bmk["Flag"]=="RETAIL")]
saldos_activos = data_bmk[(data_bmk["Flag"]=="RETAIL")]

saldos_activos = saldos_activos[saldos_activos.FechaApertura>="2024-04-01"]
cols_use = ["Cuenta","FechaApertura","Moneda","Flag","Plazo","Capital","FlagStatus"]
saldos_activos = saldos_activos[cols_use].reset_index()

# Encaje por Liquidez
# encaje_mn = 0.31/100
# encaje_me = 0.03/100
encaje_mn = 0
encaje_me = 0
saldos_activos["Encaje Liquidez"] = saldos_activos["Moneda"].apply(lambda x: encaje_mn if x=="PEN" else encaje_me)


## **2.1. Proposed Model**

In [23]:
# Dia Conductual Modelamiento
met_propuesta = saldos_activos.merge(data_model_prop.rename(columns={"Interpolated_rmst":"Cond_Day_Model"}),on=["Moneda","Flag","Plazo"])
met_propuesta["Cond_Day_Model"] = met_propuesta["Cond_Day_Model"].astype("int")

In [24]:
# TT
met_propuesta["TT"] = met_propuesta.apply(lambda x: get_tt_asignada(x["FechaApertura"],x["Moneda"],x["Cond_Day_Model"]),axis=1)

# TT Asignada
met_propuesta["TT_Asignada"] = np.power((1+met_propuesta["TT"]+met_propuesta["Encaje Liquidez"]),mes.day/360)-1

# Asignado
met_propuesta["Asignado"] = met_propuesta["TT_Asignada"]*met_propuesta["Capital"]

In [32]:
met_propuesta

,index,Cuenta,FechaApertura,Moneda,Flag,Plazo,Capital,FlagStatus,Encaje Liquidez,Cond_Day_Model,TT,TT_Asignada,Asignado
0,1126492,6013481723,2024-04-01,PEN,RETAIL,222,388.00,1,0,190,0.058444444444444445,0.004903,1.902411
1,1126493,6015845544,2024-04-01,PEN,RETAIL,182,11161.16,1,0,155,0.058916666666666666,0.004942,55.155316
2,1126494,10000000193575,2024-04-01,PEN,RETAIL,31,126.09,1,0,27,0.0611,0.00512,0.645578
3,1126495,10000000201411,2024-04-01,PEN,RETAIL,31,251.22,1,0,27,0.0611,0.00512,1.286241
4,1126496,10000000260957,2024-04-01,PEN,RETAIL,181,140.35,1,0,154,0.05893333333333334,0.004943,0.693761
...,...,...,...,...,...,...,...,...,...,...,...,...,...
263092,1412682,10000007720236,2025-03-31,PEN,RETAIL,90,12000.00,0,0,76,0.046,0.00388,46.562581
263093,1412692,22220024128015,2025-03-31,PEN,RETAIL,181,642.70,0,0,154,0.047,0.003963,2.546906
263094,1412693,22220207282011,2025-03-31,PEN,RETAIL,181,2956.92,0,0,154,0.047,0.003963,11.71775
263095,1412694,22220564751048,2025-03-31,PEN,RETAIL,19,4479.60,0,0,17,0.04686666666666667,0.003952,17.702539


In [25]:
asignado_propuesta = met_propuesta.groupby(["Moneda","Flag"],as_index=False).agg({"Asignado":"sum"})
asignado_propuesta["Asignado Solarizado"] = asignado_propuesta.apply(lambda x: x["Asignado"]*3.6680 if x["Moneda"]=="USD" else x["Asignado"],axis=1)
print(f"Asignado Total (Soles): {asignado_propuesta['Asignado Solarizado'].sum()}")
asignado_propuesta

Asignado Total (Soles): 63646493.40095932


,Moneda,Flag,Asignado,Asignado Solarizado
0,PEN,RETAIL,60116327.574546,6.011633e+07
1,USD,RETAIL,962422.526285,3.530166e+06


## **2.2. Current Model**

In [26]:
# Dia Conductual Modelamiento
met_vigente = saldos_activos.merge(data_model_vig.rename(columns={"Interpolated_rmst":"Cond_Day_Model"}),on=["Moneda","Flag","Plazo"])
met_vigente["Cond_Day_Model"] = met_vigente["Cond_Day_Model"].astype("int")

In [27]:
# TT
met_vigente["TT"] = met_vigente.apply(lambda x: get_tt_asignada(x["FechaApertura"],x["Moneda"],x["Cond_Day_Model"]),axis=1)

# TT Asignada
met_vigente["TT_Asignada"] = np.power((1+met_vigente["TT"]+met_vigente["Encaje Liquidez"]),mes.day/360)-1

# Asignado
met_vigente["Asignado"] = met_vigente["TT_Asignada"]*met_vigente["Capital"]

In [30]:
met_vigente.Capital.sum()

np.float64(13829080931.070004)

In [28]:
asignado_vigente = met_vigente.groupby(["Moneda","Flag"],as_index=False).agg({"Asignado":"sum"})
asignado_vigente["Asignado Solarizado"] = asignado_vigente.apply(lambda x: x["Asignado"]*3.6680 if x["Moneda"]=="USD" else x["Asignado"],axis=1)
print(f"Asignado Total (Soles): {asignado_vigente['Asignado Solarizado'].sum()}")
asignado_vigente

Asignado Total (Soles): 63688922.534049116


,Moneda,Flag,Asignado,Asignado Solarizado
0,PEN,RETAIL,60157214.434589,6.015721e+07
1,USD,RETAIL,962842.99331,3.531708e+06


## **2.3. Final**

In [21]:
asignado_final = pd.concat([asignado_vigente.rename(columns={"Asignado Solarizado":"Asignado Total Vigente"}),asignado_propuesta.rename(columns={"Asignado Solarizado":"Asignado Total Vigente"})],axis=1)

asignado_final = asignado_vigente[["Moneda","Asignado Solarizado"]].merge(asignado_propuesta[["Moneda","Asignado Solarizado"]],on=["Moneda"],suffixes=(" Vigente"," Propuesta"))
print(f"Asignado Total Vigente: {asignado_final['Asignado Solarizado Vigente'].sum()}")
print(f"Asignado Total Propuesta: {asignado_final['Asignado Solarizado Propuesta'].sum()}")
print(f"Diferencia Propuesta vs Vigente: {asignado_final['Asignado Solarizado Propuesta'].sum() - asignado_final['Asignado Solarizado Vigente'].sum()}")

asignado_final

Asignado Total Vigente: 3878714.8844789974
Asignado Total Propuesta: 3873105.439654327
Diferencia Propuesta vs Vigente: -5609.444824670441


,Moneda,Asignado Solarizado Vigente,Asignado Solarizado Propuesta
0,PEN,3.621603e+06,3.616112e+06
1,USD,2.571114e+05,2.569929e+05


## **2.4. Otros**

In [19]:
# # Categorías de Plazo contractual
# bins_grupo = [0, 30, 60, 90, 120, 180, 360, 720, 1080, float('inf')]
# # Etiquetas para cada grupo
# labels_grupo = [
#     "01. [0 - 30]",
#     "02. [31 - 60]",
#     "03. [61 - 90]",
#     "04. [91 - 120]",
#     "05. [121 - 180]",
#     "06. [181 - 360]",
#     "07. [361 - 720]",
#     "08. [721 - 1080]",
#     "09. [1081 - más]"
# ]
# # Aplicar la categorización
# saldos_activos["Grupo_Plazo"] = pd.cut(saldos_activos["Plazo"],bins=bins_grupo,labels=labels_grupo,right=True)

In [20]:
# prueba = saldos_activos.groupby(["Moneda","Grupo_Plazo"],as_index=False).agg({"Capital":"sum","TT_Asignada":"mean","Asignado":"sum"})
# prueba["Asignado_v2"] = prueba["TT_Asignada"]*prueba["Capital"]
# print(prueba.Asignado_v2.sum())
# prueba